# Évaluation & Interprétabilité

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from src.preprocessing import preprocess_data, build_preprocessor, load_and_split, get_feature_names
from src.evaluate import (
    plot_roc_curves, plot_precision_recall_curves, plot_confusion_matrix,
    plot_metrics_comparison, plot_feature_importance,
    plot_permutation_importance, generate_shap_plots, save_metrics_csv,
    plot_residuals, plot_regression_metrics_comparison, compute_regression_metrics
)

## Chargement des modèles entraînés

In [ ]:
X_train, X_test, y_train, y_test, _ = load_and_split('../data/raw/customer_churn_business_dataset.csv')
preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

lr = joblib.load('../models/logistic_model.joblib')
rf = joblib.load('../models/random_forest_model.joblib')
xgb = joblib.load('../models/xgboost_model.joblib')

y_proba_lr = lr.predict_proba(X_test)[:, 1]
y_proba_rf = rf.predict_proba(X_test)[:, 1]
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

models_results = [
    {'name': 'Logistic Regression', 'y_true': y_test, 'y_proba': y_proba_lr},
    {'name': 'Random Forest', 'y_true': y_test, 'y_proba': y_proba_rf},
    {'name': 'XGBoost', 'y_true': y_test, 'y_proba': y_proba_xgb},
]

## Courbes ROC et Precision-Recall

In [ ]:
plot_roc_curves(models_results, save=True)
plot_precision_recall_curves(models_results, save=True)

## Matrice de confusion du meilleur modèle

In [ ]:
best_model = joblib.load('../models/best_model.joblib')
y_pred_best = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred_best, model_name='Best Model', save=True)

## Feature Importance & SHAP

In [ ]:
xgb_clf = xgb.named_steps['classifier']
plot_feature_importance(xgb_clf, feature_names, model_name='XGBoost', top_n=15, save=True)
generate_shap_plots(xgb_clf, X_test_proc, feature_names, save=True)

## Analyse des résidus — Régression CLV

In [ ]:
from src.preprocessing import build_preprocessor_regression, load_and_split as las2
from src.evaluate import plot_residuals

X_tr_r, X_te_r, y_tr_r, y_te_r, _ = las2('../data/raw/customer_churn_business_dataset.csv', task='regression')
try:
    rf_reg = joblib.load('../models/rf_regressor_model.joblib')
    y_pred_rf_reg = rf_reg.predict(X_te_r)
    plot_residuals(y_te_r, y_pred_rf_reg, model_name='RF Regressor', save=True)
    metrics_reg = compute_regression_metrics('RF Regressor', y_te_r, y_pred_rf_reg)
    print(metrics_reg)
except FileNotFoundError:
    print("Lancez python src/train_regression.py d'abord.")

## Conclusion — Choix du modèle final

### Classification (Churn)

| Critère | Logistic Reg. | Random Forest | XGBoost | MLP |
|---|---|---|---|---|
| ROC-AUC | ~0.78 | ~0.87 | ~0.90 | ~0.86 |
| F1-Score | ~0.50 | ~0.65 | ~0.72 | ~0.62 |
| Interprétabilité | Haute | Moyenne | Moyenne (SHAP) | Faible |
| Temps train | Très rapide | Modéré | Modéré | Long |

**Recommandation** : XGBoost offre le meilleur compromis performance/interprétabilité.

### Régression (CLV)

Le MLP et le RF Regressor capturent bien les non-linéarités. Le RF est préféré pour sa robustesse et l'interprétabilité via SHAP.

### Action business
- Clients avec probabilité de churn > 0.7 : contact prioritaire par l'équipe CRM
- Revenue_at_risk = monthly_fee × proba_churn → priorisation financière
- Variables clés identifiées par SHAP : payment_failures, contract_type, csat_score, last_login_days_ago